<a href="https://colab.research.google.com/github/fergordillo880109/chatbot-taekwondo/blob/main/Chatbot_MDK_Tuchtlan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asistente de información MDK Tuchtlán

**Desarrollado por Fernanda Gómez Gordillo · Python y NLTK · Versión 1.5**

Este asistente responde consultas sobre la escuela de taekwondo MDK Tuchtlán: horarios, grupos, costos, promociones, inscripción, uniforme y equipo de protección.

## Instrucciones de uso

1. Ejecutar las celdas en orden mediante **Entorno de ejecución → Ejecutar todas**.
2. Localizar la interfaz de conversación en el **módulo 6**.
3. Escribir una consulta en el cuadro de texto y presionar **Enviar**.
4. Para comenzar otra sesión, presionar **Nueva conversación**.
5. Para conservar el registro de la sesión, seleccionar **Descargar conversación**.

## Indicaciones para la revisión

El selector de fecha permite evaluar las promociones correspondientes a distintos periodos. Después de modificar la fecha, se debe presionar **Nueva conversación** para aplicarla.

El **módulo 7** contiene las pruebas funcionales automatizadas y sus resultados.

## Funcionamiento

El chatbot utiliza reglas para identificar las consultas y seleccionar sus respuestas. Conserva un contexto limitado sobre el tipo de alumno, el mes consultado y la edad cuando esta se solicita.

Las consultas que requieren atención personalizada se orientan al WhatsApp de la escuela.

La base de conocimiento reúne los datos de la escuela, los costos y las condiciones de las promociones.

Las promociones se encuentran delimitadas por periodo. La información se actualiza manualmente y debe mantenerse consistente con las respuestas del módulo 5.

## Preparación del entorno
Instala las bibliotecas; no se requieren claves ni descargas de corpus de NLTK.

In [ ]:
%pip -q install "nltk>=3.8,<4" "ipywidgets>=8,<9"
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

## Módulo 1 — Configuración e importaciones

In [ ]:
"""Chatbot educativo basado en reglas. Python 3.10+ y NLTK."""
import re
import json
import html
import unicodedata
from datetime import date, datetime
from pathlib import Path
from nltk.tokenize import RegexpTokenizer



## Módulo 2 — Base de conocimiento editable

Estos datos se actualizan manualmente. La promoción de noviembre–diciembre de 2026 no se extiende a otros años. Los precios regulares y las reglas también se reflejan en las respuestas del módulo 5; al modificarlos, revisa y ejecuta las pruebas.

In [ ]:
ESCUELA = {
    'nombre': 'MDK Tuchtlán',
    'direccion': 'Boulevard Paso Limón #1621, colonia Paso Limón, Tuxtla Gutiérrez, Chiapas, a un costado del OXXO',
    'whatsapp': '961-654-4587',
    'enlace_whatsapp': 'https://wa.me/529616544587',
    'costos': {'inscripcion': 300, 'mensualidad': 600, 'uniforme': 1000,
               'examen_grado': 750, 'cinta_kup': 135, 'cuello_kup': 90,
               'reforzamiento_preescolar': 300},
    # Edad excepcional de “2.8 meses” pendiente de aclarar. No automatizar como admisión confirmada.
    'edad_regular': 3,
    'horarios': '¡Claro! Tenemos clases de lunes a jueves: 5:00–6:00 p. m., Preescolar e Infantil Menor (aprox. 3–9 años); 6:00–7:00 p. m., Infantil Mayor y Juveniles (aprox. 9 años en adelante). Te ayudamos a elegir el grupo según la edad y el nivel del alumno; a los 9 años valoramos cuál de los dos turnos le conviene.',
    'promocion_octubre': {'inicio': '2026-10-01', 'fin': '2026-10-31', 'descuento_uniforme': 0.20},
    'promocion_nov_dic': {'inicio': '2026-11-01', 'fin': '2026-12-31',
                         'descuento_uniforme': 0.25, 'inscripcion_gratis': False},
}
CONTACTO = f"Escríbenos por WhatsApp al {ESCUELA['whatsapp']} y con gusto te atenderemos personalmente."
TOKENIZADOR = RegexpTokenizer(r'\w+')



## Módulo 3 — Preparación del mensaje con NLTK

In [ ]:

def normalizar(texto):
    """NLTK separa tokens sin descargar corpus; conserva números y palabras."""
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto.lower())
                    if unicodedata.category(c) != 'Mn')
    return ' '.join(TOKENIZADOR.tokenize(texto))



## Módulo 4 — Reglas de reconocimiento de intenciones

In [ ]:
# Se permiten varias intenciones en un mensaje; orden fijo y respuestas sin azar.
PATRONES = {
    'reingreso': r'\b(reingreso|regresar|regreso|regresa|volver|baja|exalumno)\b',
    'promocion': r'\b(promocion|promociones|descuento|gratis|gratuito|gratuita|2x1|octubre|noviembre|diciembre|enero)\b',
    'ubicacion': r'\b(donde|ubicacion|direccion|domicilio)\b|como (?:puedo |podemos )?llegar\b',
    'contacto': r'\b(whatsapp|telefono|contacto|numero|comunicar|humano|persona)\b',
    'horarios': r'\b(horario|horarios|dias|hora|turno|turnos|sabado|sabados|domingo|domingos|viernes)\b',
    'visita': r'\b(?:puedo|podemos|podria|podriamos) (?:ir|asistir|llegar|llevar|visitar)\b|que (?:necesito|debo|tengo que) (?:para )?(?:llevar|asistir|ir)\b|primera clase|primer dia',
    'fin_semana': r'\b(sabado|sabados|domingo|domingos|sabatino|sabatinos|dominical|dominicales)\b|fin(?:es)? de semana|weekend',
    'edades': r'\b(edad|edades|anos|preescolar|preescolares|adultos|adulto|nina|ninas|nino|ninos|juvenil|juveniles|infantil|infantiles)\b',
    'documentos': r'\b(documentos|documentacion|requisitos|acta|ine|fotografias|papeles)\b|que (?:necesito|necesita|necesitan|piden|se necesita|debo llevar|tengo que llevar) para (?:inscribirme|inscribirse|inscribir|ingresar|entrar)\b',
    'clase_muestra': r'\b(prueba|muestra|probar|cita|reservar|reserva)\b',
    'uniforme': r'\b(uniforme|dobok|ropa|pants|playera|logos|logotipos)\b',
    'proteccion': r'\b(proteccion|protecciones|equipo|amarilla|careta|peto)\b',
    'experiencia': r'\b(principiante|principiantes|avanzado|avanzados|experiencia|constancia)\b|otra escuela|nunca he practicado',
    'evaluaciones': r'\b(examen|examenes|kup|reforzamiento|grado|cinta|cuello)\b',
    'mensualidad': r'\b(mensualidad|colegiatura|mensualidades|mes)\b|al mes',
    'inscripcion': r'\b(inscripcion|inscribirme|inscribirse|inscribir|inscribo|ingresar)\b',
    'costos': r'\b(costo|costos|precio|precios|tarifa|tarifas)\b|cuanto (?:cuesta|cuestan|cobran|vale|valen|pago|pagaria|debo|es|son)',
    'descripcion_clases': r'como (?:son|es|funcionan|funciona) (?:las |la |sus |tus )?clases?\b|que (?:hacen|ensenan|aprenden|se aprende|se hace) (?:en |durante )?(?:las |la |sus )?clases?\b|en que consisten (?:las |sus )?clases\b',
    'identidad': r'que es mdk|que es la escuela|asociacion|moo duk',
    'ayuda': r'\b(ayuda|menu|informes|informacion)\b',
}
REGLAS = {k: re.compile(v) for k, v in PATRONES.items()}



## Módulo 5 — Motor de conversación y respuestas

In [ ]:
class ChatbotMDK:
    def __init__(self, fecha=None):
        # Fecha explícita para reproducir las pruebas. La interfaz usa fecha actual de México.
        self.fecha = fecha or date.today()
        self.historial = []
        self.tipo_alumno = None
        self.mes_consulta = None
        self.cerrado = False

    def promocion(self, texto):
        c = ESCUELA['costos']
        if self.tipo_alumno == 'reingreso':
            return f"La promoción es exclusiva de nuevo ingreso. En reingreso se paga la colegiatura vigente: ${c['mensualidad']:,} MXN al mes. ¡Nos dará gusto recibirte de nuevo!"
        if self.mes_consulta in (11, 12):
            gratis = 'diciembre de 2026' if self.mes_consulta == 11 else 'enero de 2027'
            uniforme = round(c['uniforme'] * (1-ESCUELA['promocion_nov_dic']['descuento_uniforme']))
            total = c['inscripcion'] + c['mensualidad'] + uniforme
            return (f"Para NUEVO INGRESO en noviembre y diciembre de 2026: inscripción ${c['inscripcion']:,}, primera mensualidad ${c['mensualidad']:,} y la siguiente gratis; uniforme con 25 % de descuento: ${uniforme:,}. Suma de estos conceptos: ${total:,} MXN. Si el primer mes pagado es {'noviembre' if self.mes_consulta == 11 else 'diciembre'}, el mes gratis es {gratis}. Tienes una semana desde la inscripción para comprar el uniforme conservando la promoción. No aplica a reingresos.")
        if self.mes_consulta == 10:
            return '¡En octubre de 2026 tu inscripción es gratis y tienes 20 % de descuento en el uniforme! El uniforme queda en $800 y la mensualidad es de $600. La suma de uniforme y primera mensualidad es $1,400 MXN. Tienes una semana desde tu inscripción para comprar el uniforme conservando la promoción.'
        return '¡Con gusto te orientamos sobre las promociones para la fecha que te interesa! ' + CONTACTO

    def responder(self, mensaje):
        t = normalizar(mensaje)
        if re.fullmatch(r'\d{1,2}(?: anos)?', t) and self.historial and 'que edad' in normalizar(self.historial[-1]['respuesta']):
            t = re.search(r'\d+', t).group() + ' anos'
        if not t:
            return {'intenciones': ['vacio'], 'respuesta': 'Escribe una pregunta para comenzar.'}
        if self.cerrado:
            return {'intenciones': ['cerrado'], 'respuesta': 'La conversación terminó. Pulsa Nueva conversación para comenzar otra.'}
        if re.fullmatch(r'(salir|adios|hasta luego|terminar|cerrar)', t):
            self.cerrado = True
            return self._guardar(mensaje, ['despedida'], 'Gracias por consultar MDK Tuchtlán. ¡Hasta pronto!')
        if re.search(r'\b(reingreso|regresar|regreso|regresa|volver|baja|exalumno)\b', t):
            self.tipo_alumno = 'reingreso'
        elif re.search(r'nuevo ingreso|primera vez|soy nuevo|es nuevo', t):
            self.tipo_alumno = 'nuevo'
        meses = {'octubre':10, 'noviembre':11, 'diciembre':12, 'enero':1}
        encontrados = [v for k,v in meses.items() if re.search(r'\b'+k+r'\b', t)]
        if encontrados:
            self.mes_consulta = encontrados[0]
        elif self.mes_consulta is None:
            self.mes_consulta = self.fecha.month if self.fecha.year == 2026 else None
        intenciones = [k for k,r in REGLAS.items() if r.search(t)]
        if re.fullmatch(r'(hola|buenas|buenos dias|buenas tardes|buenas noches)',t):
            intenciones = ['saludo']
        # Evitar respuestas genéricas duplicadas cuando hay temas específicos.
        if 'fin_semana' in intenciones:
            intenciones = [i for i in intenciones if i not in ('horarios', 'visita')]
        if 'costos' in intenciones and not re.search(r'cuantos anos|desde que edad|a que edad', t):
            intenciones = [i for i in intenciones if i != 'edades']
        if 'visita' in intenciones and not re.search(r'edad|anos|adulto', t):
            intenciones = [i for i in intenciones if i != 'edades']
        if 'ubicacion' in intenciones and re.search(r'como (?:puedo |podemos )?llegar', t):
            intenciones = [i for i in intenciones if i != 'visita']
        if 'clase_muestra' in intenciones and not re.search(r'promocion|descuento|noviembre|diciembre|2x1',t):
            intenciones = [i for i in intenciones if i != 'promocion']
        if 'uniforme' in intenciones or 'evaluaciones' in intenciones or 'mensualidad' in intenciones or 'proteccion' in intenciones:
            intenciones = [i for i in intenciones if i != 'costos']
        if 'documentos' in intenciones and not re.search(r'costo|precio|cuanto|cobran', t):
            intenciones = [i for i in intenciones if i != 'inscripcion']
        if 'proteccion' in intenciones and not re.search(r'examen|kup|grado|cuello',t):
            intenciones = [i for i in intenciones if i != 'evaluaciones']
        if not intenciones and t in ('y en diciembre','y diciembre','y en noviembre','y noviembre'):
            intenciones = ['promocion']
        if not intenciones:
            intenciones = ['desconocida']
        c = ESCUELA['costos']
        respuestas = {
            'saludo': '¡Hola! Bienvenido a MDK Tuchtlán. Con gusto te ayudamos a conocer nuestros horarios, clases, costos y promociones. ¿Qué te gustaría saber?',
            'descripcion_clases': '¡Nuestras clases son de taekwondo y recibimos alumnos principiantes y avanzados! Tenemos grupos preescolares, infantiles y juveniles, y te ayudamos a elegir el grupo según la edad y el nivel. Promovemos la disciplina, el respeto y la constancia. ¡Puedes conocernos con una clase muestra gratuita! ¿Qué edad tiene quien desea asistir?',
            'identidad': '¡Somos MDK Tuchtlán! Una escuela de taekwondo de la Asociación Mexicana MDK, sistema Moo Duk Kwan. Promovemos la disciplina, el respeto y la constancia.',
            'horarios': ESCUELA['horarios'],
            'fin_semana': 'Por el momento, nuestras clases regulares son de lunes a jueves; no tenemos clases regulares los sábados ni domingos. Tenemos dos horarios: de 5:00 a 6:00 p. m. para Preescolar e Infantil Menor, y de 6:00 a 7:00 p. m. para Infantil Mayor y Juveniles. ¡Con gusto te ayudamos a elegir el grupo según la edad y el nivel!',
            'ubicacion': '¡Nos encontramos en '+ESCUELA['direccion']+'. ¡Te esperamos!',
            'contacto': CONTACTO,
            'edades': '¡Sí! Tenemos clases para niñas y niños desde los 3 años, en grupos preescolares, infantiles y juveniles. ¿Qué edad tiene tu hija o hijo? Así te orientamos sobre su grupo.',
            'visita': '¡Nos encantará recibirlos! Tenemos clases de lunes a jueves: de 5:00 a 6:00 p. m. para Preescolar e Infantil Menor, y de 6:00 a 7:00 p. m. para Infantil Mayor y Juveniles. Para conocer las clases pueden asistir a una clase muestra gratuita con pants y playera. Escríbenos por WhatsApp al 961-654-4587 para coordinar su visita y elegir el horario según la edad y el nivel.',
            'documentos': 'Para integrar tu expediente te pedimos copia del acta de nacimiento, copia de la INE del tutor y dos fotografías tamaño infantil. ¡Puedes inscribirte y entregar estos documentos después! Escríbenos por WhatsApp al 961-654-4587 y te ayudamos a elegir el grupo y realizar tu inscripción.',
            'clase_muestra': '¡Sí! Tenemos una clase muestra gratuita para que nos conozcas. Puedes venir con pants y playera. Escríbenos por WhatsApp al 961-654-4587 y con gusto te ayudamos a coordinar tu visita.',
            'uniforme': f"Nuestro uniforme tiene un precio regular de ${c['uniforme']:,} MXN y se adquiere aquí en la escuela. Al inscribirte tienes una semana para comprarlo conservando la promoción aplicable; mientras tanto puedes asistir con pants y playera. Usamos el dobok oficial: no se permite uno de otro color o con otros logotipos. ¡Pregunta por la promoción del mes en que te inscribirás!",
            'proteccion': 'Para entrenar, el equipo de protección es obligatorio a partir de cinta amarilla y se adquiere en nuestra escuela. Escríbenos por WhatsApp al 961-654-4587 y te ayudamos con los precios y el equipo que necesitas.',
            'experiencia': '¡Claro! Recibimos alumnos principiantes y avanzados. Si ya practicaste taekwondo y tienes constancia de tu último grado adquirido, ¡te lo respetamos!',
            'evaluaciones': 'Nuestros costos de evaluación son: examen de cambio de grado $750 MXN; cinta Kup $135; cuello Kup $90; y examen de reforzamiento para preescolares $300. Para conocer las próximas fechas y lo que corresponde a tu grado, escríbenos por WhatsApp al 961-654-4587.',
            'mensualidad': f"Nuestra mensualidad regular es de ${c['mensualidad']:,} MXN. Los exámenes y artículos Kup tienen un costo aparte. ¡También puedes preguntarnos por las promociones de nuevo ingreso!",
            'inscripcion': f"¡Nos encantará recibirte! La inscripción regular cuesta ${c['inscripcion']:,} MXN. Te ayudamos a elegir el grupo según la edad y el nivel del alumno. Escríbenos por WhatsApp al 961-654-4587 para acompañarte con tu inscripción y conocer la promoción aplicable.",
            'costos': '¡Claro! Nuestras clases tienen una mensualidad de $600 MXN. La inscripción regular cuesta $300 y el uniforme $1,000. La suma de estos tres conceptos es $1,900 antes de promociones. ¡Pregunta por los beneficios del mes en que quieres ingresar!',
            'ayuda': '¡Con gusto te ayudamos! Puedes preguntar por horarios, edades, dirección, costos, documentos, clase muestra, uniforme, equipo de protección y promociones. ¿Por dónde te gustaría empezar?',
            'desconocida': '¡Con gusto te ayudamos! ¿Tu consulta es sobre horarios, costos, inscripción o clases? También puedes escribirnos por WhatsApp al 961-654-4587 para recibir atención personal.',
        }
        if 'edades' in intenciones:
            if re.search(r'\badultos?\b', t):
                respuestas['edades'] = '¡Con gusto te orientamos sobre las opciones para adultos! '+CONTACTO
            else:
                edad_match = re.search(r'\b(\d+) anos\b', t)
                if edad_match and int(edad_match.group(1)) < 3:
                    respuestas['edades'] = 'Si tu pequeño aún no cumple 3 años, escríbenos por WhatsApp al 961-654-4587 para valorar su integración mediante una clase de prueba. ¡Con gusto te orientamos!'
                elif edad_match:
                    edad = int(edad_match.group(1))
                    if 3 <= edad < 9:
                        respuestas['edades'] = f'¡Para {edad} años, el grupo de Preescolar e Infantil Menor puede ser una buena opción! Sus clases son de lunes a jueves, de 5:00 a 6:00 p. m. Elegimos el grupo según la edad y el nivel. ¡Pueden conocernos con una clase muestra gratuita!'
                    elif edad == 9:
                        respuestas['edades'] = '¡A los 9 años valoramos el grupo según su edad y nivel! Tenemos el turno de 5:00 a 6:00 p. m. y el de 6:00 a 7:00 p. m., de lunes a jueves. Escríbenos por WhatsApp al 961-654-4587 para orientarte.'
                    elif edad < 18:
                        respuestas['edades'] = '¡Tenemos el grupo de Infantil Mayor y Juveniles de lunes a jueves, de 6:00 a 7:00 p. m.! Te ayudamos a elegir el grupo según la edad y el nivel. ¡Puedes conocernos con una clase muestra gratuita!'
                    else:
                        respuestas['edades'] = '¡Con gusto te orientamos sobre las opciones para adultos! '+CONTACTO
        if 'reingreso' in intenciones:
            respuestas['reingreso'] = 'Si te diste de baja y regresas, pagas la colegiatura vigente: $600 MXN al mes. La promoción de nuevo ingreso no aplica a reingresos.'
        if 'promocion' in intenciones:
            # Año ajeno no activa promociones de 2026.
            anos = re.findall(r'\b20\d{2}\b', t)
            if anos and any(a != '2026' for a in anos) and not ('enero' in t and '2027' in anos):
                respuestas['promocion'] = 'Para conocer las promociones de ese año, '+CONTACTO[0].lower()+CONTACTO[1:]
            else:
                respuestas['promocion'] = self.promocion(t)
        salida = '\n\n'.join(dict.fromkeys(respuestas[i] for i in intenciones))
        return self._guardar(mensaje, intenciones, salida)

    def _guardar(self, mensaje, intenciones, respuesta):
        fila = {'turno': len(self.historial)+1, 'entrada': mensaje,
                'intenciones': intenciones, 'respuesta': respuesta,
                'fecha_consulta': self.fecha.isoformat()}
        self.historial.append(fila)
        return fila



## Módulo 6 — Interfaz de chat dentro de Colab

In [ ]:

def mostrar_chat():
    import ipywidgets as w
    from IPython.display import display, HTML
    from zoneinfo import ZoneInfo
    hoy = datetime.now(ZoneInfo('America/Mexico_City')).date()
    fecha = w.DatePicker(description='Fecha:', value=hoy)
    bot = ChatbotMDK(hoy)
    texto = w.Textarea(placeholder='Escribe tu pregunta…', layout=w.Layout(width='100%',height='65px'))
    salida = w.Output(layout=w.Layout(border='1px solid #ddd',padding='12px',height='360px',overflow='auto'))
    enviar = w.Button(description='Enviar',button_style='success')
    nuevo = w.Button(description='Nueva conversación')
    evidencia = w.Button(description='Descargar conversación')
    aviso = w.HTML('¡Bienvenido! Conoce nuestras clases y encuentra el grupo ideal para ti.')
    def pintar(autor,mensaje):
        with salida:
            display(HTML(f'<p><b>{html.escape(autor)}:</b> {html.escape(mensaje).replace(chr(10),"<br>")}</p>'))
    def mandar(_):
        if fecha.value is None:
            pintar('Asistente','Selecciona una fecha válida.');return
        if fecha.value != bot.fecha:
            pintar('Asistente','Cambiaste la fecha. Pulsa Nueva conversación para aplicarla.');return
        consulta=texto.value
        if not consulta.strip():return
        pintar('Tú',consulta); pintar('Asistente',bot.responder(consulta)['respuesta']);texto.value=''
    def reiniciar(_):
        nonlocal bot
        bot=ChatbotMDK(fecha.value or hoy);salida.clear_output();pintar('Asistente','¡Hola! Bienvenido a MDK Tuchtlán. ¿En qué podemos ayudarte?')
    def descargar(_):
        archivo=Path('conversacion_mdk.json')
        archivo.write_text(json.dumps(bot.historial,ensure_ascii=False,indent=2),encoding='utf-8')
        try:
            from google.colab import files
            files.download(str(archivo))
        except ImportError:
            from IPython.display import FileLink
            with salida:display(FileLink(str(archivo)))
    enviar.on_click(mandar);nuevo.on_click(reiniciar);evidencia.on_click(descargar)
    display(w.VBox([w.HTML('<h2>MDK Tuchtlán · Asistente de información</h2>'),aviso,fecha,salida,texto,w.HBox([enviar,nuevo,evidencia])]))
    reiniciar(None)
    return bot  # La sesión activa la administra la interfaz; las pruebas usan instancias independientes.


In [ ]:
mostrar_chat();

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Módulo 7 — Pruebas funcionales

Este módulo evalúa el chatbot mediante quince conversaciones con consultas sobre horarios, edades, costos, inscripción, promociones, visitas y clases en fin de semana.

Las pruebas utilizan fechas simuladas y comparan las respuestas obtenidas con los resultados esperados. También incluyen consultas fuera del alcance del asistente.

Al ejecutarse, se muestran las conversaciones y el resumen de comprobaciones. Las transcripciones y los resultados se guardan en la carpeta `evidencias`.

Los resultados corresponden a los casos evaluados y no representan una medida de precisión para cualquier pregunta posible.

In [ ]:
"""Diez conversaciones reproducibles; verifican contenido y condiciones de negocio."""
from datetime import date
import json
from pathlib import Path

CASOS = [
 ('Clases en fin de semana',date(2026,10,24), [('¿Tienen clases en fin de semana?','no tenemos clases regulares los sábados ni domingos'),('¿Hay clases los sábados?','lunes a jueves'),('¿Puedo ir este domingo?','no tenemos clases regulares'),('¿Manejan grupos sabatinos?','5:00 a 6:00'),('¿Y los fines de semana?','no tenemos clases regulares')]),
 ('Conversación completa de la nueva captura',date(2026,10,24), [('hola\ntiene clase para niñas?','niñas y niños'),('5','5:00 a 6:00'),('que costo tiene las clases para preescolar?','$600'),('en noviembre que promocion tiene?','$1,650'),('que necesito para llevar a mi hijo?','pants y playera'),('donde se ubica?','Nos encontramos'),('puedo llegar este lunes?','coordinar su visita')]),
 ('Visita y ubicación separadas',date(2026,10,24), [('¿Cómo puedo llegar?','Paso Limón'),('¿Podemos asistir el martes?','lunes a jueves'),('¿Qué debo llevar para inscribirme?','acta de nacimiento')]),
 ('Consultas naturales de la captura',date(2026,10,24), [('Hola','Bienvenido'),('como son las clases?','taekwondo'),('que costo tienen las clases?','$600'),('que horarios tienen?','lunes a jueves')]),
 ('Variantes de preguntas',date(2026,10,24), [('¿Cuál es el precio de las clases?','$600'),('¿Cuánto cobran?','$600'),('¿Qué aprenden en las clases?','taekwondo'),('¿Cómo funcionan sus clases?','principiantes'),('¿Cuánto cuesta el uniforme?','$1,000')]),
 ('Ubicación y contacto',date(2026,10,24), [('Hola','Hola'),('¿DÓNDE están?','Paso Limón'),('WhatsApp','961-654-4587'),('Salir','Hasta pronto')]),
 ('Horario con edad límite',date(2026,10,24), [('¿Qué horarios tienen?','lunes a jueves'),('Mi hija tiene 9 años','edad y nivel'),('¿Qué turno le toca?','a los 9 años')]),
 ('Menor de tres años',date(2026,10,24), [('Mi hijo tiene 2 años y 8 meses','aún no cumple 3 años'),('¿Se puede reservar la clase muestra?','coordinar tu visita')]),
 ('Precios regulares',date(2026,10,24), [('¿Cuáles son sus precios?','$1,900'),('¿Cuánto cuesta al mes?','$600'),('¿Cuánto cuesta el examen y la cinta Kup?','$750')]),
 ('Documentos posteriores',date(2026,10,24), [('Quiero inscribirme','$300'),('que necesito para inscribirme?','acta de nacimiento'),('cual es el costo?','$600'),('¿Tengo que llevar el acta hoy?','entregar estos documentos después'),('¿Cuántas fotografías?','dos fotografías')]),
 ('Clase y uniforme',date(2026,10,24), [('¿La clase muestra es gratis?','gratuita'),('¿Puedo ir con pants?','una semana'),('¿Puedo llevar dobok de otra escuela con otros logos?','no se permite')]),
 ('Promoción noviembre',date(2026,11,2), [('¿Qué promoción hay para nuevo ingreso?','$1,650'),('¿Qué mes es gratis?','diciembre de 2026'),('¿Y en diciembre?','enero de 2027')]),
 ('Reingreso',date(2026,12,1), [('Me di de baja y quiero regresar','no aplica'),('¿Tengo promoción?','exclusiva de nuevo ingreso'),('¿Y si es nuevo ingreso en diciembre?','enero de 2027')]),
 ('Experiencia y protección',date(2026,10,24), [('Soy principiante','principiantes'),('Tengo constancia de otra escuela','te lo respetamos'),('¿Desde cinta amarilla necesito protección?','obligatorio'),('¿Cuánto cuesta el equipo?','precios y el equipo')]),
 ('Fuera de alcance y vigencia',date(2027,2,1), [('¿Venden pizza?','atención personal'),('¿Qué promoción hay?','fecha que te interesa'),('¿Qué promoción hay en diciembre de 2028?','ese año'),('salir','Hasta pronto'),('hola','terminó')]),
]

def ejecutar_pruebas(directorio='evidencias'):
    evidencias=[]; total=0;correctas=0
    assert normalizar('¿HORÁRIOS?')=='horarios'
    for nombre,fecha,turnos in CASOS:
        bot=ChatbotMDK(fecha);registros=[]
        for pregunta,esperado in turnos:
            resultado=bot.responder(pregunta)
            ok=esperado.lower() in resultado['respuesta'].lower()
            total+=1;correctas+=int(ok)
            registros.append({**resultado,'fragmento_esperado':esperado,'correcto':ok})
        evidencias.append({'caso':nombre,'fecha_simulada':fecha.isoformat(),'turnos':registros})
    ruta=Path(directorio);ruta.mkdir(exist_ok=True)
    resumen={'tipo':'Pruebas automáticas locales con fechas simuladas; no son capturas de Colab ni evaluación con clientes.',
             'conversaciones':len(CASOS),'mensajes':total,'correctos':correctas,
             'porcentaje':round(correctas/total*100,2),'evidencias':evidencias}
    (ruta/'resultados.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
    lineas=[resumen['tipo'],f'Conversaciones: {len(CASOS)} | Mensajes: {total} | Correctos: {correctas}']
    for caso in evidencias:
        lineas.extend(['',caso['caso']+' | '+caso['fecha_simulada']])
        for r in caso['turnos']:lineas.extend(['Usuario: '+r.get('entrada','[sesión cerrada]'),'Chatbot: '+r['respuesta'],'Validación: '+str(r['correcto'])])
    (ruta/'conversaciones.txt').write_text('\n'.join(lineas),encoding='utf-8')
    print(f'{len(CASOS)} conversaciones, {total} mensajes, {correctas} comprobaciones correctas.')
    errores=[(c['caso'],r.get('entrada'),r['respuesta']) for c in evidencias for r in c['turnos'] if not r['correcto']]
    if errores:raise AssertionError(errores)
    return resumen



resultado_pruebas = ejecutar_pruebas()
for caso in resultado_pruebas["evidencias"]:
    print("\nCONVERSACIÓN:", caso["caso"])
    for turno in caso["turnos"]:
        print("Usuario:", turno.get("entrada", "[sesión cerrada]"))
        print("Chatbot:", turno["respuesta"])

15 conversaciones, 59 mensajes, 59 comprobaciones correctas.

CONVERSACIÓN: Clases en fin de semana
Usuario: ¿Tienen clases en fin de semana?
Chatbot: Por el momento, nuestras clases regulares son de lunes a jueves; no tenemos clases regulares los sábados ni domingos. Tenemos dos horarios: de 5:00 a 6:00 p. m. para Preescolar e Infantil Menor, y de 6:00 a 7:00 p. m. para Infantil Mayor y Juveniles. ¡Con gusto te ayudamos a elegir el grupo según la edad y el nivel!
Usuario: ¿Hay clases los sábados?
Chatbot: Por el momento, nuestras clases regulares son de lunes a jueves; no tenemos clases regulares los sábados ni domingos. Tenemos dos horarios: de 5:00 a 6:00 p. m. para Preescolar e Infantil Menor, y de 6:00 a 7:00 p. m. para Infantil Mayor y Juveniles. ¡Con gusto te ayudamos a elegir el grupo según la edad y el nivel!
Usuario: ¿Puedo ir este domingo?
Chatbot: Por el momento, nuestras clases regulares son de lunes a jueves; no tenemos clases regulares los sábados ni domingos. Tenemos do

## Módulo 8 — Exportar para GitHub
Pulsa el botón para descargar un ZIP con los scripts, README, dependencias y resultados de las pruebas. Si editas los módulos, actualiza los archivos exportados de este bloque antes de publicar: el ZIP conserva el código base de esta versión. Sube también este cuaderno al repositorio. No se publica nada automáticamente.

In [ ]:
ARCHIVOS_BASE = {'chatbot_mdk.py': '# MÓDULO 1 — Configuración e importaciones\n"""Chatbot educativo basado en reglas. Python 3.10+ y NLTK."""\nimport re\nimport json\nimport html\nimport unicodedata\nfrom datetime import date, datetime\nfrom pathlib import Path\nfrom nltk.tokenize import RegexpTokenizer\n\n# MÓDULO 2 — Base de conocimiento editable\nESCUELA = {\n    \'nombre\': \'MDK Tuchtlán\',\n    \'direccion\': \'Boulevard Paso Limón #1621, colonia Paso Limón, Tuxtla Gutiérrez, Chiapas, a un costado del OXXO\',\n    \'whatsapp\': \'961-654-4587\',\n    \'enlace_whatsapp\': \'https://wa.me/529616544587\',\n    \'costos\': {\'inscripcion\': 300, \'mensualidad\': 600, \'uniforme\': 1000,\n               \'examen_grado\': 750, \'cinta_kup\': 135, \'cuello_kup\': 90,\n               \'reforzamiento_preescolar\': 300},\n    # Edad excepcional de “2.8 meses” pendiente de aclarar. No automatizar como admisión confirmada.\n    \'edad_regular\': 3,\n    \'horarios\': \'¡Claro! Tenemos clases de lunes a jueves: 5:00–6:00 p. m., Preescolar e Infantil Menor (aprox. 3–9 años); 6:00–7:00 p. m., Infantil Mayor y Juveniles (aprox. 9 años en adelante). Te ayudamos a elegir el grupo según la edad y el nivel del alumno; a los 9 años valoramos cuál de los dos turnos le conviene.\',\n    \'promocion_octubre\': {\'inicio\': \'2026-10-01\', \'fin\': \'2026-10-31\', \'descuento_uniforme\': 0.20},\n    \'promocion_nov_dic\': {\'inicio\': \'2026-11-01\', \'fin\': \'2026-12-31\',\n                         \'descuento_uniforme\': 0.25, \'inscripcion_gratis\': False},\n}\nCONTACTO = f"Escríbenos por WhatsApp al {ESCUELA[\'whatsapp\']} y con gusto te atenderemos personalmente."\nTOKENIZADOR = RegexpTokenizer(r\'\\w+\')\n\n# MÓDULO 3 — Preparación del mensaje con NLTK\n\ndef normalizar(texto):\n    """NLTK separa tokens sin descargar corpus; conserva números y palabras."""\n    texto = \'\'.join(c for c in unicodedata.normalize(\'NFD\', texto.lower())\n                    if unicodedata.category(c) != \'Mn\')\n    return \' \'.join(TOKENIZADOR.tokenize(texto))\n\n# MÓDULO 4 — Reglas de reconocimiento de intenciones\n# Se permiten varias intenciones en un mensaje; orden fijo y respuestas sin azar.\nPATRONES = {\n    \'reingreso\': r\'\\b(reingreso|regresar|regreso|regresa|volver|baja|exalumno)\\b\',\n    \'promocion\': r\'\\b(promocion|promociones|descuento|gratis|gratuito|gratuita|2x1|octubre|noviembre|diciembre|enero)\\b\',\n    \'ubicacion\': r\'\\b(donde|ubicacion|direccion|domicilio)\\b|como (?:puedo |podemos )?llegar\\b\',\n    \'contacto\': r\'\\b(whatsapp|telefono|contacto|numero|comunicar|humano|persona)\\b\',\n    \'horarios\': r\'\\b(horario|horarios|dias|hora|turno|turnos|sabado|sabados|domingo|domingos|viernes)\\b\',\n    \'visita\': r\'\\b(?:puedo|podemos|podria|podriamos) (?:ir|asistir|llegar|llevar|visitar)\\b|que (?:necesito|debo|tengo que) (?:para )?(?:llevar|asistir|ir)\\b|primera clase|primer dia\',\n    \'fin_semana\': r\'\\b(sabado|sabados|domingo|domingos|sabatino|sabatinos|dominical|dominicales)\\b|fin(?:es)? de semana|weekend\',\n    \'edades\': r\'\\b(edad|edades|anos|preescolar|preescolares|adultos|adulto|nina|ninas|nino|ninos|juvenil|juveniles|infantil|infantiles)\\b\',\n    \'documentos\': r\'\\b(documentos|documentacion|requisitos|acta|ine|fotografias|papeles)\\b|que (?:necesito|necesita|necesitan|piden|se necesita|debo llevar|tengo que llevar) para (?:inscribirme|inscribirse|inscribir|ingresar|entrar)\\b\',\n    \'clase_muestra\': r\'\\b(prueba|muestra|probar|cita|reservar|reserva)\\b\',\n    \'uniforme\': r\'\\b(uniforme|dobok|ropa|pants|playera|logos|logotipos)\\b\',\n    \'proteccion\': r\'\\b(proteccion|protecciones|equipo|amarilla|careta|peto)\\b\',\n    \'experiencia\': r\'\\b(principiante|principiantes|avanzado|avanzados|experiencia|constancia)\\b|otra escuela|nunca he practicado\',\n    \'evaluaciones\': r\'\\b(examen|examenes|kup|reforzamiento|grado|cinta|cuello)\\b\',\n    \'mensualidad\': r\'\\b(mensualidad|colegiatura|mensualidades|mes)\\b|al mes\',\n    \'inscripcion\': r\'\\b(inscripcion|inscribirme|inscribirse|inscribir|inscribo|ingresar)\\b\',\n    \'costos\': r\'\\b(costo|costos|precio|precios|tarifa|tarifas)\\b|cuanto (?:cuesta|cuestan|cobran|vale|valen|pago|pagaria|debo|es|son)\',\n    \'descripcion_clases\': r\'como (?:son|es|funcionan|funciona) (?:las |la |sus |tus )?clases?\\b|que (?:hacen|ensenan|aprenden|se aprende|se hace) (?:en |durante )?(?:las |la |sus )?clases?\\b|en que consisten (?:las |sus )?clases\\b\',\n    \'identidad\': r\'que es mdk|que es la escuela|asociacion|moo duk\',\n    \'ayuda\': r\'\\b(ayuda|menu|informes|informacion)\\b\',\n}\nREGLAS = {k: re.compile(v) for k, v in PATRONES.items()}\n\n# MÓDULO 5 — Motor de conversación y respuestas\nclass ChatbotMDK:\n    def __init__(self, fecha=None):\n        # Fecha explícita para reproducir las pruebas. La interfaz usa fecha actual de México.\n        self.fecha = fecha or date.today()\n        self.historial = []\n        self.tipo_alumno = None\n        self.mes_consulta = None\n        self.cerrado = False\n\n    def promocion(self, texto):\n        c = ESCUELA[\'costos\']\n        if self.tipo_alumno == \'reingreso\':\n            return f"La promoción es exclusiva de nuevo ingreso. En reingreso se paga la colegiatura vigente: ${c[\'mensualidad\']:,} MXN al mes. ¡Nos dará gusto recibirte de nuevo!"\n        if self.mes_consulta in (11, 12):\n            gratis = \'diciembre de 2026\' if self.mes_consulta == 11 else \'enero de 2027\'\n            uniforme = round(c[\'uniforme\'] * (1-ESCUELA[\'promocion_nov_dic\'][\'descuento_uniforme\']))\n            total = c[\'inscripcion\'] + c[\'mensualidad\'] + uniforme\n            return (f"Para NUEVO INGRESO en noviembre y diciembre de 2026: inscripción ${c[\'inscripcion\']:,}, primera mensualidad ${c[\'mensualidad\']:,} y la siguiente gratis; uniforme con 25 % de descuento: ${uniforme:,}. Suma de estos conceptos: ${total:,} MXN. Si el primer mes pagado es {\'noviembre\' if self.mes_consulta == 11 else \'diciembre\'}, el mes gratis es {gratis}. Tienes una semana desde la inscripción para comprar el uniforme conservando la promoción. No aplica a reingresos.")\n        if self.mes_consulta == 10:\n            return \'¡En octubre de 2026 tu inscripción es gratis y tienes 20 % de descuento en el uniforme! El uniforme queda en $800 y la mensualidad es de $600. La suma de uniforme y primera mensualidad es $1,400 MXN. Tienes una semana desde tu inscripción para comprar el uniforme conservando la promoción.\'\n        return \'¡Con gusto te orientamos sobre las promociones para la fecha que te interesa! \' + CONTACTO\n\n    def responder(self, mensaje):\n        t = normalizar(mensaje)\n        if re.fullmatch(r\'\\d{1,2}(?: anos)?\', t) and self.historial and \'que edad\' in normalizar(self.historial[-1][\'respuesta\']):\n            t = re.search(r\'\\d+\', t).group() + \' anos\'\n        if not t:\n            return {\'intenciones\': [\'vacio\'], \'respuesta\': \'Escribe una pregunta para comenzar.\'}\n        if self.cerrado:\n            return {\'intenciones\': [\'cerrado\'], \'respuesta\': \'La conversación terminó. Pulsa Nueva conversación para comenzar otra.\'}\n        if re.fullmatch(r\'(salir|adios|hasta luego|terminar|cerrar)\', t):\n            self.cerrado = True\n            return self._guardar(mensaje, [\'despedida\'], \'Gracias por consultar MDK Tuchtlán. ¡Hasta pronto!\')\n        if re.search(r\'\\b(reingreso|regresar|regreso|regresa|volver|baja|exalumno)\\b\', t):\n            self.tipo_alumno = \'reingreso\'\n        elif re.search(r\'nuevo ingreso|primera vez|soy nuevo|es nuevo\', t):\n            self.tipo_alumno = \'nuevo\'\n        meses = {\'octubre\':10, \'noviembre\':11, \'diciembre\':12, \'enero\':1}\n        encontrados = [v for k,v in meses.items() if re.search(r\'\\b\'+k+r\'\\b\', t)]\n        if encontrados:\n            self.mes_consulta = encontrados[0]\n        elif self.mes_consulta is None:\n            self.mes_consulta = self.fecha.month if self.fecha.year == 2026 else None\n        intenciones = [k for k,r in REGLAS.items() if r.search(t)]\n        if re.fullmatch(r\'(hola|buenas|buenos dias|buenas tardes|buenas noches)\',t):\n            intenciones = [\'saludo\']\n        # Evitar respuestas genéricas duplicadas cuando hay temas específicos.\n        if \'fin_semana\' in intenciones:\n            intenciones = [i for i in intenciones if i not in (\'horarios\', \'visita\')]\n        if \'costos\' in intenciones and not re.search(r\'cuantos anos|desde que edad|a que edad\', t):\n            intenciones = [i for i in intenciones if i != \'edades\']\n        if \'visita\' in intenciones and not re.search(r\'edad|anos|adulto\', t):\n            intenciones = [i for i in intenciones if i != \'edades\']\n        if \'ubicacion\' in intenciones and re.search(r\'como (?:puedo |podemos )?llegar\', t):\n            intenciones = [i for i in intenciones if i != \'visita\']\n        if \'clase_muestra\' in intenciones and not re.search(r\'promocion|descuento|noviembre|diciembre|2x1\',t):\n            intenciones = [i for i in intenciones if i != \'promocion\']\n        if \'uniforme\' in intenciones or \'evaluaciones\' in intenciones or \'mensualidad\' in intenciones or \'proteccion\' in intenciones:\n            intenciones = [i for i in intenciones if i != \'costos\']\n        if \'documentos\' in intenciones and not re.search(r\'costo|precio|cuanto|cobran\', t):\n            intenciones = [i for i in intenciones if i != \'inscripcion\']\n        if \'proteccion\' in intenciones and not re.search(r\'examen|kup|grado|cuello\',t):\n            intenciones = [i for i in intenciones if i != \'evaluaciones\']\n        if not intenciones and t in (\'y en diciembre\',\'y diciembre\',\'y en noviembre\',\'y noviembre\'):\n            intenciones = [\'promocion\']\n        if not intenciones:\n            intenciones = [\'desconocida\']\n        c = ESCUELA[\'costos\']\n        respuestas = {\n            \'saludo\': \'¡Hola! Bienvenido a MDK Tuchtlán. Con gusto te ayudamos a conocer nuestros horarios, clases, costos y promociones. ¿Qué te gustaría saber?\',\n            \'descripcion_clases\': \'¡Nuestras clases son de taekwondo y recibimos alumnos principiantes y avanzados! Tenemos grupos preescolares, infantiles y juveniles, y te ayudamos a elegir el grupo según la edad y el nivel. Promovemos la disciplina, el respeto y la constancia. ¡Puedes conocernos con una clase muestra gratuita! ¿Qué edad tiene quien desea asistir?\',\n            \'identidad\': \'¡Somos MDK Tuchtlán! Una escuela de taekwondo de la Asociación Mexicana MDK, sistema Moo Duk Kwan. Promovemos la disciplina, el respeto y la constancia.\',\n            \'horarios\': ESCUELA[\'horarios\'],\n            \'fin_semana\': \'Por el momento, nuestras clases regulares son de lunes a jueves; no tenemos clases regulares los sábados ni domingos. Tenemos dos horarios: de 5:00 a 6:00 p. m. para Preescolar e Infantil Menor, y de 6:00 a 7:00 p. m. para Infantil Mayor y Juveniles. ¡Con gusto te ayudamos a elegir el grupo según la edad y el nivel!\',\n            \'ubicacion\': \'¡Nos encontramos en \'+ESCUELA[\'direccion\']+\'. ¡Te esperamos!\',\n            \'contacto\': CONTACTO,\n            \'edades\': \'¡Sí! Tenemos clases para niñas y niños desde los 3 años, en grupos preescolares, infantiles y juveniles. ¿Qué edad tiene tu hija o hijo? Así te orientamos sobre su grupo.\',\n            \'visita\': \'¡Nos encantará recibirlos! Tenemos clases de lunes a jueves: de 5:00 a 6:00 p. m. para Preescolar e Infantil Menor, y de 6:00 a 7:00 p. m. para Infantil Mayor y Juveniles. Para conocer las clases pueden asistir a una clase muestra gratuita con pants y playera. Escríbenos por WhatsApp al 961-654-4587 para coordinar su visita y elegir el horario según la edad y el nivel.\',\n            \'documentos\': \'Para integrar tu expediente te pedimos copia del acta de nacimiento, copia de la INE del tutor y dos fotografías tamaño infantil. ¡Puedes inscribirte y entregar estos documentos después! Escríbenos por WhatsApp al 961-654-4587 y te ayudamos a elegir el grupo y realizar tu inscripción.\',\n            \'clase_muestra\': \'¡Sí! Tenemos una clase muestra gratuita para que nos conozcas. Puedes venir con pants y playera. Escríbenos por WhatsApp al 961-654-4587 y con gusto te ayudamos a coordinar tu visita.\',\n            \'uniforme\': f"Nuestro uniforme tiene un precio regular de ${c[\'uniforme\']:,} MXN y se adquiere aquí en la escuela. Al inscribirte tienes una semana para comprarlo conservando la promoción aplicable; mientras tanto puedes asistir con pants y playera. Usamos el dobok oficial: no se permite uno de otro color o con otros logotipos. ¡Pregunta por la promoción del mes en que te inscribirás!",\n            \'proteccion\': \'Para entrenar, el equipo de protección es obligatorio a partir de cinta amarilla y se adquiere en nuestra escuela. Escríbenos por WhatsApp al 961-654-4587 y te ayudamos con los precios y el equipo que necesitas.\',\n            \'experiencia\': \'¡Claro! Recibimos alumnos principiantes y avanzados. Si ya practicaste taekwondo y tienes constancia de tu último grado adquirido, ¡te lo respetamos!\',\n            \'evaluaciones\': \'Nuestros costos de evaluación son: examen de cambio de grado $750 MXN; cinta Kup $135; cuello Kup $90; y examen de reforzamiento para preescolares $300. Para conocer las próximas fechas y lo que corresponde a tu grado, escríbenos por WhatsApp al 961-654-4587.\',\n            \'mensualidad\': f"Nuestra mensualidad regular es de ${c[\'mensualidad\']:,} MXN. Los exámenes y artículos Kup tienen un costo aparte. ¡También puedes preguntarnos por las promociones de nuevo ingreso!",\n            \'inscripcion\': f"¡Nos encantará recibirte! La inscripción regular cuesta ${c[\'inscripcion\']:,} MXN. Te ayudamos a elegir el grupo según la edad y el nivel del alumno. Escríbenos por WhatsApp al 961-654-4587 para acompañarte con tu inscripción y conocer la promoción aplicable.",\n            \'costos\': \'¡Claro! Nuestras clases tienen una mensualidad de $600 MXN. La inscripción regular cuesta $300 y el uniforme $1,000. La suma de estos tres conceptos es $1,900 antes de promociones. ¡Pregunta por los beneficios del mes en que quieres ingresar!\',\n            \'ayuda\': \'¡Con gusto te ayudamos! Puedes preguntar por horarios, edades, dirección, costos, documentos, clase muestra, uniforme, equipo de protección y promociones. ¿Por dónde te gustaría empezar?\',\n            \'desconocida\': \'¡Con gusto te ayudamos! ¿Tu consulta es sobre horarios, costos, inscripción o clases? También puedes escribirnos por WhatsApp al 961-654-4587 para recibir atención personal.\',\n        }\n        if \'edades\' in intenciones:\n            if re.search(r\'\\badultos?\\b\', t):\n                respuestas[\'edades\'] = \'¡Con gusto te orientamos sobre las opciones para adultos! \'+CONTACTO\n            else:\n                edad_match = re.search(r\'\\b(\\d+) anos\\b\', t)\n                if edad_match and int(edad_match.group(1)) < 3:\n                    respuestas[\'edades\'] = \'Si tu pequeño aún no cumple 3 años, escríbenos por WhatsApp al 961-654-4587 para valorar su integración mediante una clase de prueba. ¡Con gusto te orientamos!\'\n                elif edad_match:\n                    edad = int(edad_match.group(1))\n                    if 3 <= edad < 9:\n                        respuestas[\'edades\'] = f\'¡Para {edad} años, el grupo de Preescolar e Infantil Menor puede ser una buena opción! Sus clases son de lunes a jueves, de 5:00 a 6:00 p. m. Elegimos el grupo según la edad y el nivel. ¡Pueden conocernos con una clase muestra gratuita!\'\n                    elif edad == 9:\n                        respuestas[\'edades\'] = \'¡A los 9 años valoramos el grupo según su edad y nivel! Tenemos el turno de 5:00 a 6:00 p. m. y el de 6:00 a 7:00 p. m., de lunes a jueves. Escríbenos por WhatsApp al 961-654-4587 para orientarte.\'\n                    elif edad < 18:\n                        respuestas[\'edades\'] = \'¡Tenemos el grupo de Infantil Mayor y Juveniles de lunes a jueves, de 6:00 a 7:00 p. m.! Te ayudamos a elegir el grupo según la edad y el nivel. ¡Puedes conocernos con una clase muestra gratuita!\'\n                    else:\n                        respuestas[\'edades\'] = \'¡Con gusto te orientamos sobre las opciones para adultos! \'+CONTACTO\n        if \'reingreso\' in intenciones:\n            respuestas[\'reingreso\'] = \'Si te diste de baja y regresas, pagas la colegiatura vigente: $600 MXN al mes. La promoción de nuevo ingreso no aplica a reingresos.\'\n        if \'promocion\' in intenciones:\n            # Año ajeno no activa promociones de 2026.\n            anos = re.findall(r\'\\b20\\d{2}\\b\', t)\n            if anos and any(a != \'2026\' for a in anos) and not (\'enero\' in t and \'2027\' in anos):\n                respuestas[\'promocion\'] = \'Para conocer las promociones de ese año, \'+CONTACTO[0].lower()+CONTACTO[1:]\n            else:\n                respuestas[\'promocion\'] = self.promocion(t)\n        salida = \'\\n\\n\'.join(dict.fromkeys(respuestas[i] for i in intenciones))\n        return self._guardar(mensaje, intenciones, salida)\n\n    def _guardar(self, mensaje, intenciones, respuesta):\n        fila = {\'turno\': len(self.historial)+1, \'entrada\': mensaje,\n                \'intenciones\': intenciones, \'respuesta\': respuesta,\n                \'fecha_consulta\': self.fecha.isoformat()}\n        self.historial.append(fila)\n        return fila\n\n# MÓDULO 6 — Interfaz de chat dentro de Colab\n\ndef mostrar_chat():\n    import ipywidgets as w\n    from IPython.display import display, HTML\n    from zoneinfo import ZoneInfo\n    hoy = datetime.now(ZoneInfo(\'America/Mexico_City\')).date()\n    fecha = w.DatePicker(description=\'Fecha:\', value=hoy)\n    bot = ChatbotMDK(hoy)\n    texto = w.Textarea(placeholder=\'Escribe tu pregunta…\', layout=w.Layout(width=\'100%\',height=\'65px\'))\n    salida = w.Output(layout=w.Layout(border=\'1px solid #ddd\',padding=\'12px\',height=\'360px\',overflow=\'auto\'))\n    enviar = w.Button(description=\'Enviar\',button_style=\'success\')\n    nuevo = w.Button(description=\'Nueva conversación\')\n    evidencia = w.Button(description=\'Descargar conversación\')\n    aviso = w.HTML(\'¡Bienvenido! Conoce nuestras clases y encuentra el grupo ideal para ti.\')\n    def pintar(autor,mensaje):\n        with salida:\n            display(HTML(f\'<p><b>{html.escape(autor)}:</b> {html.escape(mensaje).replace(chr(10),"<br>")}</p>\'))\n    def mandar(_):\n        if fecha.value is None:\n            pintar(\'Asistente\',\'Selecciona una fecha válida.\');return\n        if fecha.value != bot.fecha:\n            pintar(\'Asistente\',\'Cambiaste la fecha. Pulsa Nueva conversación para aplicarla.\');return\n        consulta=texto.value\n        if not consulta.strip():return\n        pintar(\'Tú\',consulta); pintar(\'Asistente\',bot.responder(consulta)[\'respuesta\']);texto.value=\'\'\n    def reiniciar(_):\n        nonlocal bot\n        bot=ChatbotMDK(fecha.value or hoy);salida.clear_output();pintar(\'Asistente\',\'¡Hola! Bienvenido a MDK Tuchtlán. ¿En qué podemos ayudarte?\')\n    def descargar(_):\n        archivo=Path(\'conversacion_mdk.json\')\n        archivo.write_text(json.dumps(bot.historial,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n        try:\n            from google.colab import files\n            files.download(str(archivo))\n        except ImportError:\n            from IPython.display import FileLink\n            with salida:display(FileLink(str(archivo)))\n    enviar.on_click(mandar);nuevo.on_click(reiniciar);evidencia.on_click(descargar)\n    display(w.VBox([w.HTML(\'<h2>MDK Tuchtlán · Asistente de información</h2>\'),aviso,fecha,salida,texto,w.HBox([enviar,nuevo,evidencia])]))\n    reiniciar(None)\n    return bot  # La sesión activa la administra la interfaz; las pruebas usan instancias independientes.\n', 'pruebas_mdk.py': '"""Diez conversaciones reproducibles; verifican contenido y condiciones de negocio."""\nfrom datetime import date\nimport json\nfrom pathlib import Path\nfrom chatbot_mdk import ChatbotMDK, normalizar\n\nCASOS = [\n (\'Clases en fin de semana\',date(2026,10,24), [(\'¿Tienen clases en fin de semana?\',\'no tenemos clases regulares los sábados ni domingos\'),(\'¿Hay clases los sábados?\',\'lunes a jueves\'),(\'¿Puedo ir este domingo?\',\'no tenemos clases regulares\'),(\'¿Manejan grupos sabatinos?\',\'5:00 a 6:00\'),(\'¿Y los fines de semana?\',\'no tenemos clases regulares\')]),\n (\'Conversación completa de la nueva captura\',date(2026,10,24), [(\'hola\\ntiene clase para niñas?\',\'niñas y niños\'),(\'5\',\'5:00 a 6:00\'),(\'que costo tiene las clases para preescolar?\',\'$600\'),(\'en noviembre que promocion tiene?\',\'$1,650\'),(\'que necesito para llevar a mi hijo?\',\'pants y playera\'),(\'donde se ubica?\',\'Nos encontramos\'),(\'puedo llegar este lunes?\',\'coordinar su visita\')]),\n (\'Visita y ubicación separadas\',date(2026,10,24), [(\'¿Cómo puedo llegar?\',\'Paso Limón\'),(\'¿Podemos asistir el martes?\',\'lunes a jueves\'),(\'¿Qué debo llevar para inscribirme?\',\'acta de nacimiento\')]),\n (\'Consultas naturales de la captura\',date(2026,10,24), [(\'Hola\',\'Bienvenido\'),(\'como son las clases?\',\'taekwondo\'),(\'que costo tienen las clases?\',\'$600\'),(\'que horarios tienen?\',\'lunes a jueves\')]),\n (\'Variantes de preguntas\',date(2026,10,24), [(\'¿Cuál es el precio de las clases?\',\'$600\'),(\'¿Cuánto cobran?\',\'$600\'),(\'¿Qué aprenden en las clases?\',\'taekwondo\'),(\'¿Cómo funcionan sus clases?\',\'principiantes\'),(\'¿Cuánto cuesta el uniforme?\',\'$1,000\')]),\n (\'Ubicación y contacto\',date(2026,10,24), [(\'Hola\',\'Hola\'),(\'¿DÓNDE están?\',\'Paso Limón\'),(\'WhatsApp\',\'961-654-4587\'),(\'Salir\',\'Hasta pronto\')]),\n (\'Horario con edad límite\',date(2026,10,24), [(\'¿Qué horarios tienen?\',\'lunes a jueves\'),(\'Mi hija tiene 9 años\',\'edad y nivel\'),(\'¿Qué turno le toca?\',\'a los 9 años\')]),\n (\'Menor de tres años\',date(2026,10,24), [(\'Mi hijo tiene 2 años y 8 meses\',\'aún no cumple 3 años\'),(\'¿Se puede reservar la clase muestra?\',\'coordinar tu visita\')]),\n (\'Precios regulares\',date(2026,10,24), [(\'¿Cuáles son sus precios?\',\'$1,900\'),(\'¿Cuánto cuesta al mes?\',\'$600\'),(\'¿Cuánto cuesta el examen y la cinta Kup?\',\'$750\')]),\n (\'Documentos posteriores\',date(2026,10,24), [(\'Quiero inscribirme\',\'$300\'),(\'que necesito para inscribirme?\',\'acta de nacimiento\'),(\'cual es el costo?\',\'$600\'),(\'¿Tengo que llevar el acta hoy?\',\'entregar estos documentos después\'),(\'¿Cuántas fotografías?\',\'dos fotografías\')]),\n (\'Clase y uniforme\',date(2026,10,24), [(\'¿La clase muestra es gratis?\',\'gratuita\'),(\'¿Puedo ir con pants?\',\'una semana\'),(\'¿Puedo llevar dobok de otra escuela con otros logos?\',\'no se permite\')]),\n (\'Promoción noviembre\',date(2026,11,2), [(\'¿Qué promoción hay para nuevo ingreso?\',\'$1,650\'),(\'¿Qué mes es gratis?\',\'diciembre de 2026\'),(\'¿Y en diciembre?\',\'enero de 2027\')]),\n (\'Reingreso\',date(2026,12,1), [(\'Me di de baja y quiero regresar\',\'no aplica\'),(\'¿Tengo promoción?\',\'exclusiva de nuevo ingreso\'),(\'¿Y si es nuevo ingreso en diciembre?\',\'enero de 2027\')]),\n (\'Experiencia y protección\',date(2026,10,24), [(\'Soy principiante\',\'principiantes\'),(\'Tengo constancia de otra escuela\',\'te lo respetamos\'),(\'¿Desde cinta amarilla necesito protección?\',\'obligatorio\'),(\'¿Cuánto cuesta el equipo?\',\'precios y el equipo\')]),\n (\'Fuera de alcance y vigencia\',date(2027,2,1), [(\'¿Venden pizza?\',\'atención personal\'),(\'¿Qué promoción hay?\',\'fecha que te interesa\'),(\'¿Qué promoción hay en diciembre de 2028?\',\'ese año\'),(\'salir\',\'Hasta pronto\'),(\'hola\',\'terminó\')]),\n]\n\ndef ejecutar_pruebas(directorio=\'evidencias\'):\n    evidencias=[]; total=0;correctas=0\n    assert normalizar(\'¿HORÁRIOS?\')==\'horarios\'\n    for nombre,fecha,turnos in CASOS:\n        bot=ChatbotMDK(fecha);registros=[]\n        for pregunta,esperado in turnos:\n            resultado=bot.responder(pregunta)\n            ok=esperado.lower() in resultado[\'respuesta\'].lower()\n            total+=1;correctas+=int(ok)\n            registros.append({**resultado,\'fragmento_esperado\':esperado,\'correcto\':ok})\n        evidencias.append({\'caso\':nombre,\'fecha_simulada\':fecha.isoformat(),\'turnos\':registros})\n    ruta=Path(directorio);ruta.mkdir(exist_ok=True)\n    resumen={\'tipo\':\'Pruebas automáticas locales con fechas simuladas; no son capturas de Colab ni evaluación con clientes.\',\n             \'conversaciones\':len(CASOS),\'mensajes\':total,\'correctos\':correctas,\n             \'porcentaje\':round(correctas/total*100,2),\'evidencias\':evidencias}\n    (ruta/\'resultados.json\').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    lineas=[resumen[\'tipo\'],f\'Conversaciones: {len(CASOS)} | Mensajes: {total} | Correctos: {correctas}\']\n    for caso in evidencias:\n        lineas.extend([\'\',caso[\'caso\']+\' | \'+caso[\'fecha_simulada\']])\n        for r in caso[\'turnos\']:lineas.extend([\'Usuario: \'+r.get(\'entrada\',\'[sesión cerrada]\'),\'Chatbot: \'+r[\'respuesta\'],\'Validación: \'+str(r[\'correcto\'])])\n    (ruta/\'conversaciones.txt\').write_text(\'\\n\'.join(lineas),encoding=\'utf-8\')\n    print(f\'{len(CASOS)} conversaciones, {total} mensajes, {correctas} comprobaciones correctas.\')\n    errores=[(c[\'caso\'],r.get(\'entrada\'),r[\'respuesta\']) for c in evidencias for r in c[\'turnos\'] if not r[\'correcto\']]\n    if errores:raise AssertionError(errores)\n    return resumen\n\nif __name__==\'__main__\':ejecutar_pruebas()\n', 'README.md': '# Asistente MDK Tuchtlán\n\nPrototipo académico de un chatbot basado en reglas, desarrollado con Python 3.10+, NLTK e ipywidgets. Atiende información de la escuela y conserva un contexto mínimo de tipo de alumno, mes consultado y respuesta breve de edad. No utiliza React, servicios de IA generativa ni entrenamiento.\n\n## Uso en Google Colab\n1. Abre Colab y selecciona Archivo > Subir cuaderno; carga Chatbot_MDK_Tuchtlan.ipynb.\n2. Selecciona Entorno de ejecución > Ejecutar todas. La primera celda instala dependencias.\n3. Ve al módulo 6: escribe preguntas en el cuadro y pulsa Enviar.\n4. Para simular noviembre o diciembre de 2026, cambia la fecha y pulsa Nueva conversación.\n5. Descarga la conversación para conservar evidencia. El módulo 7 ejecuta quince conversaciones automáticas y el módulo 8 exporta el paquete para GitHub.\n\n## Ejecución local\nInstala requirements.txt con pip y ejecuta `python pruebas_mdk.py`. Para la interfaz, utiliza el cuaderno Jupyter o Colab. El módulo Python no inicia un servidor web.\n\n## Organización\n- Módulos 1–2: configuración y base de conocimiento.\n- Módulos 3–4: normalización con RegexpTokenizer de NLTK y patrones de intención.\n- Módulo 5: motor de reglas y contexto mínimo.\n- Módulo 6: interfaz ipywidgets dentro del cuaderno.\n- Módulo 7: pruebas automáticas y evidencias exportables.\n- Módulo 8: exportación de código, README y resultados para GitHub.\n\n## Información y reglas\nFuentes: materiales de MDK Tuchtlán e información proporcionada por la responsable a través de Fernanda Gómez Gordillo en octubre de 2026. Costos expresados en MXN. Promoción de octubre de 2026: inscripción gratis y 20 % de descuento en uniforme ($800); mensualidad $600, suma $1,400. Dirección del anuncio: Boulevard Paso Limón #1621. Promoción noviembre–diciembre 2026 exclusiva de nuevo ingreso: inscripción $300, primera mensualidad $600 y siguiente gratis, uniforme $750. Si ingresa en diciembre, enero de 2027 es gratis. Reingreso paga colegiatura vigente, actualmente $600. El uniforme puede adquirirse dentro de una semana desde la inscripción respetando la promoción. Se permiten pants y playera mientras tanto. Equipo obligatorio desde cinta amarilla y comprado en la escuela.\n\n## Pendientes y límites\n- Edad excepcional escrita como “2.8 meses”: no está confirmada como 2 años y 8 meses; se remite a WhatsApp.\n- Precio y piezas del equipo, necesidad de cita, admisión de adultos sin confirmar.\n- No reserva, inscribe, cobra ni envía mensajes por WhatsApp. El enlace abre un canal de contacto que usa la persona.\n- Reconoce patrones previstos; errores ortográficos no contemplados, negaciones, cambios de tema y consultas ambiguas pueden fallar. No es un sistema de comprensión general.\n- La fecha visible permite reproducir pruebas. Las promociones posteriores a diciembre de 2026 requieren actualización manual. Revisar la base de conocimiento antes de uso real.\n- Las quince conversaciones automáticas usan fechas simuladas y comprobaciones de contenido; no equivalen a evaluación con personas ni garantizan precisión general. Conservar también capturas de las pruebas manuales en Colab.\n- El historial se mantiene en memoria durante la sesión; Nueva conversación lo borra. Descárgalo antes. No escribir datos personales de alumnos.\n\n## GitHub\nCrea un repositorio y sube el cuaderno, chatbot_mdk.py, pruebas_mdk.py, requirements.txt, README.md y evidencias de prueba. La exportación no publica ni crea el repositorio automáticamente. Revisa las evidencias antes de publicarlas.\n\n## Referencias técnicas\n- NLTK, RegexpTokenizer: https://www.nltk.org/api/nltk.tokenize.regexp.html\n- ipywidgets: https://ipywidgets.readthedocs.io/\n- Google Colab: https://colab.research.google.com/\n', 'requirements.txt': 'nltk>=3.8,<4\nipywidgets>=8,<9\n'}
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink
boton_exportar = widgets.Button(description='Descargar código y README', layout=widgets.Layout(width='260px'))
def exportar_proyecto(_):
    destino = Path('MDK_Tuchtlan_GitHub.zip')
    with zipfile.ZipFile(destino, 'w', zipfile.ZIP_DEFLATED) as z:
        for nombre, contenido in ARCHIVOS_BASE.items():
            z.writestr(nombre, contenido)
        for archivo in Path('evidencias').glob('*'):
            if archivo.is_file(): z.write(archivo, str(archivo))
    try:
        from google.colab import files
        files.download(str(destino))
    except ImportError:
        display(FileLink(str(destino)))
boton_exportar.on_click(exportar_proyecto)
display(boton_exportar)


Button(description='Descargar código y README', layout=Layout(width='260px'), style=ButtonStyle())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## README del proyecto

# Asistente MDK Tuchtlán

Prototipo académico de un chatbot basado en reglas, desarrollado con Python 3.10+, NLTK e ipywidgets. Atiende información de la escuela y conserva un contexto mínimo de tipo de alumno, mes consultado y respuesta breve de edad. No utiliza React, servicios de IA generativa ni entrenamiento.

## Uso en Google Colab
1. Abre Colab y selecciona Archivo > Subir cuaderno; carga Chatbot_MDK_Tuchtlan.ipynb.
2. Selecciona Entorno de ejecución > Ejecutar todas. La primera celda instala dependencias.
3. Ve al módulo 6: escribe preguntas en el cuadro y pulsa Enviar.
4. Para simular noviembre o diciembre de 2026, cambia la fecha y pulsa Nueva conversación.
5. Descarga la conversación para conservar evidencia. El módulo 7 ejecuta quince conversaciones automáticas y el módulo 8 exporta el paquete para GitHub.

## Ejecución local
Instala requirements.txt con pip y ejecuta `python pruebas_mdk.py`. Para la interfaz, utiliza el cuaderno Jupyter o Colab. El módulo Python no inicia un servidor web.

## Organización
- Módulos 1–2: configuración y base de conocimiento.
- Módulos 3–4: normalización con RegexpTokenizer de NLTK y patrones de intención.
- Módulo 5: motor de reglas y contexto mínimo.
- Módulo 6: interfaz ipywidgets dentro del cuaderno.
- Módulo 7: pruebas automáticas y evidencias exportables.
- Módulo 8: exportación de código, README y resultados para GitHub.

## Información y reglas
Fuentes: materiales de MDK Tuchtlán e información proporcionada por la responsable a través de Fernanda Gómez Gordillo en octubre de 2026. Costos expresados en MXN. Promoción de octubre de 2026: inscripción gratis y 20 % de descuento en uniforme ($800); mensualidad $600, suma $1,400. Dirección del anuncio: Boulevard Paso Limón #1621. Promoción noviembre–diciembre 2026 exclusiva de nuevo ingreso: inscripción $300, primera mensualidad $600 y siguiente gratis, uniforme $750. Si ingresa en diciembre, enero de 2027 es gratis. Reingreso paga colegiatura vigente, actualmente $600. El uniforme puede adquirirse dentro de una semana desde la inscripción respetando la promoción. Se permiten pants y playera mientras tanto. Equipo obligatorio desde cinta amarilla y comprado en la escuela.

## Pendientes y límites
- Edad excepcional escrita como “2.8 meses”: no está confirmada como 2 años y 8 meses; se remite a WhatsApp.
- Precio y piezas del equipo, necesidad de cita, admisión de adultos sin confirmar.
- No reserva, inscribe, cobra ni envía mensajes por WhatsApp. El enlace abre un canal de contacto que usa la persona.
- Reconoce patrones previstos; errores ortográficos no contemplados, negaciones, cambios de tema y consultas ambiguas pueden fallar. No es un sistema de comprensión general.
- La fecha visible permite reproducir pruebas. Las promociones posteriores a diciembre de 2026 requieren actualización manual. Revisar la base de conocimiento antes de uso real.
- Las quince conversaciones automáticas usan fechas simuladas y comprobaciones de contenido; no equivalen a evaluación con personas ni garantizan precisión general. Conservar también capturas de las pruebas manuales en Colab.
- El historial se mantiene en memoria durante la sesión; Nueva conversación lo borra. Descárgalo antes. No escribir datos personales de alumnos.

## GitHub
Crea un repositorio y sube el cuaderno, chatbot_mdk.py, pruebas_mdk.py, requirements.txt, README.md y evidencias de prueba. La exportación no publica ni crea el repositorio automáticamente. Revisa las evidencias antes de publicarlas.

## Referencias técnicas
- NLTK, RegexpTokenizer: https://www.nltk.org/api/nltk.tokenize.regexp.html
- ipywidgets: https://ipywidgets.readthedocs.io/
- Google Colab: https://colab.research.google.com/
